# Transport: R(T), I-V, pulse sweeps

Sections (each one uses the newest run of its kind, or the run number you set; a section without a matching run
is skipped with a message):

1. **R(T) sweep** (`rt_sweep`, run_kind `R(T)`): the derived `RSample` column and where its formula comes from,
   cooling vs warming, dR/dT, transition temperature estimate
2. **R(T) steps** (`tm_rt_steps`, run_kind `R(T) steps`): the `resistance` column
3. **I-V curves** (`iv_curves`, run_kind `I-V`): the `iv` stream per curve, linear fit for R, hysteresis
4. **Pulse sweeps** (`tm_pulse_sweep`, run_kind `PulseSweep`): pulsed I-V, R per pulse

In [ ]:
%matplotlib inline
from nbsetup import *
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})

## 1. R(T) sweep

### Where `RSample` comes from

Instruments publish only what they measure (lock-in X, its output amplitude, ...). How a sample quantity
follows from them is part of the experiment, so it is a **derived value** (`control/fastq/derived.py`):
a formula with named inputs (`device.attr`) and constants, kept in `<data>/derived.json` (edited on the
dashboard's `derived` device page, seeded from `derived.values` in the lab file).

At the start of every run the definitions valid at that moment are copied into the run metadata,
`run.md["derived"]`, together with the data key under which each input is recorded. `run.table()` then
evaluates the formula per event from the raw data (inputs found in another stream, e.g. the lock-in
amplitude in `mfli1_amplitude_monitor`, are interpolated in time). So a run always uses *its own* formula, even
if the definition is changed later.

In [ ]:
RT_RUN = None   # None = newest R(T) sweep; or a scan number / uid prefix
rt = pick_run(RT_RUN, kind="R(T)", required=False)
if rt is not None:
    print(rt)
    for d in rt.md.get("derived") or []:
        print(" ", D.describe(d))
        print("     recorded inputs:", d.get("keys"), "| units:", d.get("units"), "|", d.get("doc"))
    hints = rt.md.get("plot_hints") or {}
    TCOL = hints.get("x")                                   # temperature readback of the controller
    cols = list(data_keys(rt.node))
    RCOL = "RSample" if "RSample" in cols else next(s for s in hints.get("signals", []) if s in cols)
    print(f"temperature column {TCOL!r}, resistance column {RCOL!r}, segments {rt.md.get('segments')}")

Check: evaluate the formula by hand from the raw columns, and re-evaluate it with a different constant
(e.g. the real value of the series resistor) - `D.evaluate` takes a definition and the input arrays.

In [ ]:
if rt is not None and RCOL == "RSample":
    d = next(d for d in rt.md["derived"] if d["name"] == RCOL)
    stream = max((s for s in rt.streams if RCOL in data_keys(rt.node, s)), key=lambda s: (rt.stop.get("num_events") or {}).get(s, 0))
    m = rt.table(stream)
    raw = {}
    for name, key in d["keys"].items():
        src = next(s for s in rt.streams if key in data_keys(rt.node, s) and "derived" not in data_keys(rt.node, s)[key])
        t = rt.table(src, columns=["time", key])
        raw[name] = np.interp(m["time"], t["time"], t[key])
    c = d["constants"]
    by_hand = raw["x"] / ((raw["amp"] / np.sqrt(2) - raw["x"]) / c["series_r"]) if set(raw) == {"x", "amp"} else D.evaluate(d, raw)
    print(f"{stream}: {len(m)} rows, max relative deviation table vs by hand:",
          float(np.nanmax(np.abs(by_hand / m[RCOL] - 1))))
    other = dict(d, constants={**c, **({"series_r": c["series_r"] * 1.01} if "series_r" in c else {})})
    print("with series_r + 1 %: R changes by", f"{float(np.nanmedian(D.evaluate(other, raw) / m[RCOL] - 1)) * 100:.2f} %")

### R(T): primary points and the continuous record

The `primary` stream has one reading every `period_s`; the lock-in is also **monitored** (every update,
`<signal>_monitor` streams), and so is the temperature. For a dense R(T) the densest stream with `RSample`
is taken and the temperature is interpolated onto its time stamps.

In [ ]:
def dense_rt(run, rcol, tcol):
    """time, T, R from the densest stream holding rcol, T interpolated from the temperature monitor."""
    cands = [s for s in run.streams if rcol in data_keys(run.node, s)]
    stream = max(cands, key=lambda s: (run.stop.get("num_events") or {}).get(s, 0))
    out = run.table(stream, columns=["time", rcol]).rename(columns={rcol: "R"})
    tsrc = f"{tcol}_monitor" if f"{tcol}_monitor" in run.streams else "primary"
    tt = run.table(tsrc, columns=["time", tcol]).sort_values("time")
    out["T"] = np.interp(out["time"], tt["time"], tt[tcol])
    out["t_s"] = out["time"] - run.md["time"]
    return out[["time", "t_s", "T", "R"]].dropna(), stream

if rt is not None:
    rtd, rstream = dense_rt(rt, RCOL, TCOL)
    prim = rt.table(columns=[TCOL, RCOL, "time"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(rtd["t_s"], rtd["T"], "C0-", lw=1)
    axes[0].set_xlabel("time since start (s)"); axes[0].set_ylabel(f"{TCOL} (K)", color="C0")
    ax2 = axes[0].twinx(); ax2.plot(rtd["t_s"], rtd["R"], "C1.", ms=2); ax2.set_ylabel(f"{RCOL} (Ohm)", color="C1")
    axes[1].plot(rtd["T"], rtd["R"], ".", ms=2, alpha=0.4, label=f"{rstream} ({len(rtd)})")
    axes[1].plot(prim[TCOL], prim[RCOL], "o", ms=5, label=f"primary ({len(prim)})")
    axes[1].set_xlabel(f"{TCOL} (K)"); axes[1].set_ylabel(f"{RCOL} (Ohm)"); axes[1].legend(fontsize=8)
    fig.suptitle(label(rt)); fig.tight_layout(); plt.show()

### Cooling vs warming, dR/dT, transition temperature

The sweep direction comes from the sign of dT/dt (smoothed); `MIN_RATE_K_PER_MIN` separates holds from sweeps.
Each branch is binned in temperature (`BIN_K`, None = span / 80), dR/dT is the gradient of the binned curve.
Transition estimates per branch: the temperature of the steepest |dR/dT|, and where R crosses the midpoint
between its extremes. Their difference warming − cooling is the thermal hysteresis (plus any thermometer lag).

In [ ]:
MIN_RATE_K_PER_MIN = 0.05
BIN_K = None

if rt is not None:
    s = rtd.sort_values("time").reset_index(drop=True)
    w = max(5, len(s) // 50)
    Ts = s["T"].rolling(w, center=True, min_periods=1).mean()
    rate = np.gradient(Ts, s["time"]) * 60                                   # K/min
    s["branch"] = np.where(rate < -MIN_RATE_K_PER_MIN, "cooling", np.where(rate > MIN_RATE_K_PER_MIN, "warming", "hold"))
    print(s["branch"].value_counts().to_string())
    span = s["T"].max() - s["T"].min()
    bin_k = BIN_K or max(span / 80, 1e-3)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    tc = {}
    for br, colr in (("cooling", "C0"), ("warming", "C3")):
        part = s[s["branch"] == br]
        if len(part) < 5:
            continue
        b = part.groupby((part["T"] / bin_k).round() * bin_k)["R"].agg(["mean", "count"]).query("count > 0")
        Tb, Rb = b.index.to_numpy(float), b["mean"].to_numpy(float)
        dRdT = np.gradient(Rb, Tb) if len(Tb) > 2 else np.full_like(Tb, np.nan)
        axes[0].plot(part["T"], part["R"], ".", ms=2, alpha=0.3, color=colr)
        axes[0].plot(Tb, Rb, "-", color=colr, label=br)
        axes[1].plot(Tb, dRdT, "-", color=colr, label=br)
        inner = slice(1, -1) if len(Tb) > 4 else slice(None)
        steep = Tb[inner][int(np.nanargmax(np.abs(dRdT[inner])))] if np.isfinite(dRdT).any() else np.nan
        mid = (Rb.max() + Rb.min()) / 2
        cross = np.flatnonzero(np.diff(np.sign(Rb - mid)) != 0)
        i = cross[0] if len(cross) else None                                  # first midpoint crossing
        Tmid = Tb[i] + (mid - Rb[i]) * (Tb[i + 1] - Tb[i]) / (Rb[i + 1] - Rb[i]) if i is not None else np.nan
        tc[br] = {"T_steepest_K": steep, "T_midpoint_K": Tmid, "R_min": Rb.min(), "R_max": Rb.max(), "points": len(part)}
    axes[0].set_xlabel(f"{TCOL} (K)"); axes[0].set_ylabel(f"{RCOL} (Ohm)"); axes[0].legend()
    axes[1].set_xlabel(f"{TCOL} (K)"); axes[1].set_ylabel("dR/dT (Ohm/K)"); axes[1].legend()
    fig.suptitle(f"{label(rt)} - bins of {bin_k:.3g} K"); fig.tight_layout(); plt.show()
    tc = pd.DataFrame(tc).T
    display(tc)
    if {"cooling", "warming"} <= set(tc.index):
        print("hysteresis warming - cooling:",
              f"{tc.at['warming', 'T_steepest_K'] - tc.at['cooling', 'T_steepest_K']:.3f} K (steepest),",
              f"{tc.at['warming', 'T_midpoint_K'] - tc.at['cooling', 'T_midpoint_K']:.3f} K (midpoint)")

## 2. R(T) steps (`tm_rt_steps`)

At each temperature the plan waits until the controller is stable, then the Keithley 6221/2182A measures in
Delta mode. The `resistance` column is computed by the plan: mean Delta voltage / `current_a`
(the excitation, recorded in `run.md["current_a"]`). The error bar is the standard error of the Delta readings.

In [ ]:
STEPS_RUN = None
steps = pick_run(STEPS_RUN, kind="R(T) steps", required=False)
if steps is not None:
    st = steps.table()
    k = next(d for d in steps.md.get("detectors") or [] if f"{d}_mean" in st.columns)
    Ttc = (steps.md.get("plot_hints") or {}).get("x") or (steps.md.get("roles") or {}).get("temperature")
    I = steps.md["current_a"]
    st["R_check"] = st[f"{k}_mean"] / I
    st["R_err"] = st[f"{k}_std_dev"] / np.sqrt(st[f"{k}_n"].clip(lower=1)) / abs(I)
    print(steps, f"current {I:g} A; max |resistance - mean/current| =", float((st["resistance"] - st["R_check"]).abs().max()))
    fig, ax = plt.subplots()
    ax.errorbar(st[Ttc], st["resistance"], yerr=st["R_err"], fmt="o-", capsize=3)
    ax.set_xlabel(f"{Ttc} (K)"); ax.set_ylabel("R (Ohm)"); ax.set_title(label(steps))
    plt.show()
    display(st[[Ttc, "resistance", "R_err", f"{k}_n"]].round(6))

## 3. I-V curves (`iv_curves`)

The `iv` stream holds every point (`curve`, `current_a`, `voltage_v`, `t_s`, `temperature_k`); `primary` one
summary event per curve. Per curve: a straight-line fit V = R·I + V₀ gives the resistance. Hysteresis: the
sweep goes 0 → +max → 0 → −max → 0; the up- and down-going branches are compared at equal current, and the
loop area ∮V dI (zero for an ohmic sample) is a single number for it.

In [ ]:
IV_RUN = None
iv = pick_run(IV_RUN, kind="I-V", required=False, columns=["current_a", "voltage_v"], stream="iv")
if iv is not None:
    ivt = iv.table("iv")
    trapz = getattr(np, "trapezoid", None) or np.trapz
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for n, c in ivt.groupby("curve"):
        I, V = c["current_a"].to_numpy(float), c["voltage_v"].to_numpy(float)
        (R, V0), cov = np.polyfit(I, V, 1, cov=True) if len(I) > 3 else (np.polyfit(I, V, 1), np.full((2, 2), np.nan))
        dI = np.sign(np.gradient(I))
        up, down = (dI > 0), (dI < 0)
        grid = np.linspace(max(I[up].min(), I[down].min()), min(I[up].max(), I[down].max()), 50) if up.any() and down.any() else []
        dv = (np.interp(grid, *zip(*sorted(zip(I[down], V[down])))) - np.interp(grid, *zip(*sorted(zip(I[up], V[up]))))) if len(grid) else [np.nan]
        rows.append({"curve": int(n), "T_K": c["temperature_k"].mean(), "R_ohm": R, "R_err": np.sqrt(cov[0, 0]),
                     "V0_V": V0, "max|V_down - V_up|": np.nanmax(np.abs(dv)), "loop_area_VA": abs(trapz(V, I)),
                     "points": len(c)})
        axes[0].plot(I * 1e6, V * 1e3, "o", ms=3, label=f"curve {int(n)} ({c['temperature_k'].mean():.2f} K)")
        axes[0].plot(I * 1e6, (R * I + V0) * 1e3, "-", lw=0.8, color=axes[0].lines[-1].get_color())
        axes[1].plot(I * 1e6, (V - (R * I + V0)) * 1e6, ".-", lw=0.5, label=f"curve {int(n)}")
    axes[0].set_xlabel("I (µA)"); axes[0].set_ylabel("V (mV)"); axes[0].legend(fontsize=8); axes[0].set_title(label(iv))
    axes[1].set_xlabel("I (µA)"); axes[1].set_ylabel("V − fit (µV)"); axes[1].set_title("residual of the linear fit")
    fig.tight_layout(); plt.show()
    ivres = pd.DataFrame(rows).set_index("curve")
    display(ivres)
    if len(ivres) > 1:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.errorbar(ivres["T_K"], ivres["R_ohm"], yerr=ivres["R_err"], fmt="o-", capsize=3)
        ax.set_xlabel("T (K)"); ax.set_ylabel("R from I-V (Ohm)"); plt.show()

## 4. Pulse sweeps (`tm_pulse_sweep`)

Pulsed I-V (6221 pulse-delta sweep): one point per pulse in the `iv` stream. R = V/I per pulse shows
self-heating as a rise of R with |I|; the pulse settings are in the plan call (`plan_request`).

In [ ]:
PULSE_RUN = None
ps = pick_run(PULSE_RUN, kind="PulseSweep", required=False, columns=["current_a", "voltage_v"], stream="iv")
if ps is not None:
    pt = ps.table("iv")
    kw = (ps.md.get("plan_request") or {}).get("kwargs") or {}
    print(ps, {k: kw.get(k) for k in ("spacing", "start_a", "stop_a", "step_a", "width_s", "compliance_v") if k in kw})
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for n, c in pt.groupby("curve"):
        I, V = c["current_a"].to_numpy(float), c["voltage_v"].to_numpy(float)
        nz = np.abs(I) > 0
        R, V0 = np.polyfit(I, V, 1)
        axes[0].plot(I * 1e6, V * 1e3, "o", ms=3, label=f"curve {int(n)}: R = {R:.4g} Ohm (fit)")
        axes[0].plot(I * 1e6, (R * I + V0) * 1e3, "-", lw=0.8, color=axes[0].lines[-1].get_color())
        axes[1].plot(I[nz] * 1e6, V[nz] / I[nz], "o-", ms=3, label=f"curve {int(n)}")
    axes[0].set_xlabel("I (µA)"); axes[0].set_ylabel("V (mV)"); axes[0].legend(fontsize=8); axes[0].set_title(label(ps))
    axes[1].set_xlabel("I (µA)"); axes[1].set_ylabel("V / I (Ohm)"); axes[1].set_title("resistance per pulse")
    fig.tight_layout(); plt.show()